# Capstone Part 4: Quantization

Quantization reduces model size and increases inference speed by using lower-precision
number representations. We will apply:

1. **Dynamic INT8 quantization** (PyTorch native)
2. **Static INT8 quantization** with calibration
3. **INT4 quantization** (simulated GPTQ-style)

For each, we benchmark:
- Model size on disk
- Inference speed (tokens/second)
- Quality (perplexity on eval set)

**Key insight:** Modern LLMs are often over-parameterized, meaning we can remove
significant precision with minimal quality loss. The challenge is finding the right
trade-off.

## 1. Environment Setup

In [ ]:
!pip install -q torch datasets tiktoken matplotlib tqdm

In [ ]:
import math
import time
import os
import tempfile
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model Architecture and DPO Checkpoint

In [ ]:
# ---- Model definition (same as previous notebooks) ----

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    max_seq_len: int = 512
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 688
    dropout: float = 0.1
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return (x * (x.float().pow(2).mean(-1, keepdim=True) + self.eps).rsqrt()).type_as(x) * self.weight


def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_seq_len).float(), freqs)
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x, freqs_cis):
    b, s, n, d = x.shape
    x_c = torch.view_as_complex(x.float().reshape(b, s, n, -1, 2))
    fc = freqs_cis[:s].unsqueeze(0).unsqueeze(2)
    return torch.view_as_real(x_c * fc).reshape(b, s, n, d).type_as(x)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads, self.n_kv_heads = config.n_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def _repeat_kv(self, x):
        if self.n_rep == 1: return x
        b, s, n, d = x.shape
        return x[:,:,:,None,:].expand(b,s,n,self.n_rep,d).reshape(b,s,self.n_heads,d)

    def forward(self, x, freqs_cis, mask=None):
        b, s, _ = x.shape
        q = self.wq(x).reshape(b,s,self.n_heads,self.head_dim)
        k = self.wk(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        v = self.wv(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        q, k = apply_rope(q, freqs_cis), apply_rope(k, freqs_cis)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        attn = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if mask is not None: attn = attn.masked_fill(mask == 0, float("-inf"))
        out = torch.matmul(self.attn_dropout(F.softmax(attn, dim=-1)), v)
        return self.resid_dropout(self.wo(out.transpose(1,2).reshape(b,s,-1)))


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)

    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        return x + self.ffn(self.ffn_norm(x))


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        head_dim = config.d_model // config.n_heads
        self.register_buffer("freqs_cis", precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        b, s = input_ids.shape
        mask = torch.tril(torch.ones(s, s, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        x = self.dropout(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            input_ids = torch.cat([input_ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
        return input_ids


print("Model architecture defined.")

In [ ]:
save_dir = Path("checkpoints")
dpo_checkpoint = torch.load(save_dir / "best_dpo.pt", map_location="cpu", weights_only=False)

config = dpo_checkpoint["config"]
tokenizer = tiktoken.get_encoding("gpt2")

# Load the FP32 model (on CPU for quantization)
fp32_model = MiniLLM(config)
fp32_model.load_state_dict(dpo_checkpoint["model_state_dict"])
fp32_model.eval()

print(f"Loaded DPO checkpoint (step {dpo_checkpoint['step']})")
print(f"Parameters: {sum(p.numel() for p in fp32_model.parameters()):,}")

## 3. Evaluation Dataset

We need a small eval set to measure perplexity across quantization methods.

In [ ]:
from datasets import load_dataset

# Load a small evaluation set
raw_eval = load_dataset("roneneldan/TinyStories", split="validation[:500]")

eval_tokens = []
for ex in raw_eval:
    tokens = tokenizer.encode(ex["text"])
    eval_tokens.extend(tokens)

# Create fixed-length chunks
seq_len = config.max_seq_len
eval_tensor = torch.tensor(eval_tokens[:len(eval_tokens) // (seq_len + 1) * (seq_len + 1)])
eval_chunks = eval_tensor.reshape(-1, seq_len + 1)
eval_inputs = eval_chunks[:, :-1]
eval_targets = eval_chunks[:, 1:]

print(f"Eval chunks: {eval_chunks.shape[0]}")
print(f"Total eval tokens: {eval_chunks.numel():,}")

## 4. Benchmarking Utilities

In [ ]:
def measure_model_size(model, name="model"):
    """Measure model size on disk."""
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / 1024 / 1024
        os.unlink(f.name)
    print(f"{name} size: {size_mb:.2f} MB")
    return size_mb


@torch.no_grad()
def compute_perplexity(model, eval_inputs, eval_targets, batch_size=16, device="cpu"):
    """Compute perplexity on eval set."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for i in range(0, len(eval_inputs), batch_size):
        x = eval_inputs[i:i + batch_size].to(device)
        y = eval_targets[i:i + batch_size].to(device)
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), reduction='sum')
        total_loss += loss.item()
        total_tokens += y.numel()

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)


def measure_inference_speed(model, tokenizer, device="cpu", n_runs=5, max_tokens=50):
    """Measure tokens per second during generation."""
    model.eval()
    prompt = "Once upon a time"
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

    # Warmup
    with torch.no_grad():
        _ = model.generate(input_ids.clone(), max_new_tokens=10, temperature=0.8)

    # Timed runs
    times = []
    for _ in range(n_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(input_ids.clone(), max_new_tokens=max_tokens, temperature=0.8)
        times.append(time.time() - start)

    avg_time = sum(times) / len(times)
    tokens_per_sec = max_tokens / avg_time
    return tokens_per_sec, avg_time


print("Benchmark utilities defined.")

## 5. Baseline: FP32 Model

In [ ]:
print("=" * 60)
print("Baseline: FP32 Model")
print("=" * 60)

fp32_size = measure_model_size(fp32_model, "FP32")
fp32_ppl = compute_perplexity(fp32_model, eval_inputs, eval_targets, device="cpu")
fp32_speed, fp32_time = measure_inference_speed(fp32_model, tokenizer, device="cpu")

print(f"Perplexity: {fp32_ppl:.2f}")
print(f"Inference speed: {fp32_speed:.1f} tokens/s")
print(f"Average generation time ({50} tokens): {fp32_time:.3f}s")

results = {
    "FP32": {"size_mb": fp32_size, "perplexity": fp32_ppl, "tokens_per_sec": fp32_speed},
}

## 6. Weight Distribution Analysis

Before quantizing, let's examine the weight distributions. Quantization works best
when weights are normally distributed around zero with small outliers.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Collect weight tensors from different layers
weight_info = [
    ("Embedding", fp32_model.tok_emb.weight.data),
    ("Layer 0 - Q proj", fp32_model.layers[0].attention.wq.weight.data),
    ("Layer 0 - FFN w1", fp32_model.layers[0].ffn.w1.weight.data),
    ("Layer 4 - Q proj", fp32_model.layers[4].attention.wq.weight.data),
    ("Layer 4 - FFN w1", fp32_model.layers[4].ffn.w1.weight.data),
    ("Layer 7 - FFN w2", fp32_model.layers[7].ffn.w2.weight.data),
]

for ax, (name, w) in zip(axes.flat, weight_info):
    w_flat = w.flatten().numpy()
    ax.hist(w_flat, bins=100, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f"{name}\nmean={w_flat.mean():.4f}, std={w_flat.std():.4f}")
    ax.set_xlabel('Weight Value')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Weight Distributions Before Quantization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Well-distributed weights around zero are ideal for symmetric quantization.")

## 7. Dynamic INT8 Quantization

PyTorch's dynamic quantization converts Linear layer weights to INT8 and performs
activations quantization on-the-fly during inference. This is the simplest approach:
- No calibration data needed
- Typically 2-4x size reduction
- Modest speedup on CPU

In [ ]:
import copy

# Dynamic quantization targets Linear layers
int8_dynamic_model = copy.deepcopy(fp32_model)
int8_dynamic_model = torch.ao.quantization.quantize_dynamic(
    int8_dynamic_model,
    {nn.Linear},  # Quantize all Linear layers
    dtype=torch.qint8,
)

print("=" * 60)
print("Dynamic INT8 Quantization")
print("=" * 60)

int8_dyn_size = measure_model_size(int8_dynamic_model, "INT8-Dynamic")
int8_dyn_ppl = compute_perplexity(int8_dynamic_model, eval_inputs, eval_targets, device="cpu")
int8_dyn_speed, _ = measure_inference_speed(int8_dynamic_model, tokenizer, device="cpu")

print(f"Perplexity: {int8_dyn_ppl:.2f} (FP32: {fp32_ppl:.2f}, delta: {int8_dyn_ppl - fp32_ppl:+.2f})")
print(f"Size reduction: {fp32_size / int8_dyn_size:.2f}x")
print(f"Inference speed: {int8_dyn_speed:.1f} tokens/s ({int8_dyn_speed/fp32_speed:.2f}x)")

results["INT8-Dynamic"] = {
    "size_mb": int8_dyn_size,
    "perplexity": int8_dyn_ppl,
    "tokens_per_sec": int8_dyn_speed,
}

## 8. Manual INT8 Weight-Only Quantization

To understand quantization deeply, let's implement symmetric per-channel weight
quantization from scratch. This is what happens inside GPTQ, AWQ, etc.

**Symmetric quantization** maps FP32 weights to INT8:
$$w_q = \text{round}\left(\frac{w}{s}\right), \quad s = \frac{\max(|w|)}{127}$$

**Dequantization** recovers approximate FP32:
$$\hat{w} = w_q \cdot s$$

In [ ]:
class QuantizedLinear(nn.Module):
    """INT8 weight-only quantized linear layer (per-channel symmetric)."""

    def __init__(self, weight: torch.Tensor, bits: int = 8):
        super().__init__()
        self.bits = bits
        self.qmax = 2 ** (bits - 1) - 1  # 127 for INT8, 7 for INT4

        # Per-output-channel scale
        max_val = weight.abs().amax(dim=1, keepdim=True)
        scale = max_val / self.qmax
        scale = scale.clamp(min=1e-8)  # Avoid division by zero

        # Quantize
        w_quantized = (weight / scale).round().clamp(-self.qmax, self.qmax).to(torch.int8)

        self.register_buffer("weight_quantized", w_quantized)
        self.register_buffer("scale", scale)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dequantize on-the-fly
        w_dequantized = self.weight_quantized.float() * self.scale
        return F.linear(x, w_dequantized)

    @property
    def out_features(self):
        return self.weight_quantized.shape[0]

    @property
    def in_features(self):
        return self.weight_quantized.shape[1]


# Demonstrate quantization error
test_weight = fp32_model.layers[0].attention.wq.weight.data
ql = QuantizedLinear(test_weight, bits=8)
reconstructed = ql.weight_quantized.float() * ql.scale
error = (test_weight - reconstructed).abs()
print(f"INT8 Quantization Error:")
print(f"  Max error: {error.max().item():.6f}")
print(f"  Mean error: {error.mean().item():.6f}")
print(f"  RMSE: {error.pow(2).mean().sqrt().item():.6f}")
print(f"  Relative error: {(error / test_weight.abs().clamp(min=1e-8)).mean().item():.4%}")

In [ ]:
def quantize_model_manual(model, bits=8):
    """Replace all Linear layers with quantized versions."""
    quantized = copy.deepcopy(model)

    def replace_linear(module, prefix=""):
        for name, child in module.named_children():
            full_name = f"{prefix}.{name}" if prefix else name
            if isinstance(child, nn.Linear):
                ql = QuantizedLinear(child.weight.data, bits=bits)
                setattr(module, name, ql)
            else:
                replace_linear(child, full_name)

    replace_linear(quantized)

    # Fix weight tying (lm_head needs to reference embedding)
    # Since we quantized both, we just keep them separate

    return quantized


int8_manual_model = quantize_model_manual(fp32_model, bits=8)
int8_manual_model.eval()

print("=" * 60)
print("Manual INT8 Weight-Only Quantization")
print("=" * 60)

int8_man_size = measure_model_size(int8_manual_model, "INT8-Manual")
int8_man_ppl = compute_perplexity(int8_manual_model, eval_inputs, eval_targets, device="cpu")
int8_man_speed, _ = measure_inference_speed(int8_manual_model, tokenizer, device="cpu")

print(f"Perplexity: {int8_man_ppl:.2f} (FP32: {fp32_ppl:.2f}, delta: {int8_man_ppl - fp32_ppl:+.2f})")
print(f"Inference speed: {int8_man_speed:.1f} tokens/s")

results["INT8-Manual"] = {
    "size_mb": int8_man_size,
    "perplexity": int8_man_ppl,
    "tokens_per_sec": int8_man_speed,
}

## 9. INT4 Quantization (Simulated GPTQ-Style)

INT4 is more aggressive: each weight is stored in just 4 bits, achieving ~8x compression.
Modern approaches like GPTQ use second-order information (Hessian) to minimize quantization
error. Here we implement a simplified version with per-group quantization.

**Group quantization** divides each row into groups of G values, each with its own scale.
This improves accuracy by allowing different ranges for different weight groups.

In [ ]:
class QuantizedLinear4Bit(nn.Module):
    """INT4 weight quantized linear layer with per-group scales."""

    def __init__(self, weight: torch.Tensor, group_size: int = 32):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.group_size = group_size
        self.qmax = 7  # 4-bit signed: -8 to 7

        # Pad if needed
        pad_size = (group_size - self.in_features % group_size) % group_size
        if pad_size > 0:
            weight = F.pad(weight, (0, pad_size))

        padded_in = weight.shape[1]
        n_groups = padded_in // group_size

        # Reshape for per-group quantization
        w_grouped = weight.reshape(self.out_features, n_groups, group_size)

        # Per-group scale
        max_val = w_grouped.abs().amax(dim=2, keepdim=True)
        scale = max_val / self.qmax
        scale = scale.clamp(min=1e-8)

        # Quantize to 4-bit range
        w_quantized = (w_grouped / scale).round().clamp(-8, self.qmax).to(torch.int8)

        self.register_buffer("weight_quantized", w_quantized)  # (out, n_groups, group_size)
        self.register_buffer("scale", scale)  # (out, n_groups, 1)
        self.padded_in = padded_in

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dequantize
        w = (self.weight_quantized.float() * self.scale).reshape(self.out_features, self.padded_in)
        w = w[:, :self.in_features]  # Remove padding
        return F.linear(x, w)


# Demonstrate INT4 quantization error
ql4 = QuantizedLinear4Bit(test_weight, group_size=32)
reconstructed_4 = (ql4.weight_quantized.float() * ql4.scale).reshape(ql4.out_features, ql4.padded_in)
reconstructed_4 = reconstructed_4[:, :test_weight.shape[1]]
error_4 = (test_weight - reconstructed_4).abs()

print(f"INT4 Quantization Error (group_size=32):")
print(f"  Max error: {error_4.max().item():.6f}")
print(f"  Mean error: {error_4.mean().item():.6f}")
print(f"  RMSE: {error_4.pow(2).mean().sqrt().item():.6f}")
print(f"  Relative error: {(error_4 / test_weight.abs().clamp(min=1e-8)).mean().item():.4%}")
print(f"\nCompare INT8 mean error: {error.mean().item():.6f} vs INT4: {error_4.mean().item():.6f}")

In [ ]:
def quantize_model_4bit(model, group_size=32):
    """Replace all Linear layers with 4-bit quantized versions."""
    quantized = copy.deepcopy(model)

    def replace_linear(module):
        for name, child in module.named_children():
            if isinstance(child, nn.Linear):
                setattr(module, name, QuantizedLinear4Bit(child.weight.data, group_size))
            else:
                replace_linear(child)

    replace_linear(quantized)
    return quantized


int4_model = quantize_model_4bit(fp32_model, group_size=32)
int4_model.eval()

print("=" * 60)
print("INT4 Quantization (Group Size 32)")
print("=" * 60)

int4_size = measure_model_size(int4_model, "INT4")
int4_ppl = compute_perplexity(int4_model, eval_inputs, eval_targets, device="cpu")
int4_speed, _ = measure_inference_speed(int4_model, tokenizer, device="cpu")

print(f"Perplexity: {int4_ppl:.2f} (FP32: {fp32_ppl:.2f}, delta: {int4_ppl - fp32_ppl:+.2f})")
print(f"Inference speed: {int4_speed:.1f} tokens/s")

results["INT4"] = {
    "size_mb": int4_size,
    "perplexity": int4_ppl,
    "tokens_per_sec": int4_speed,
}

## 10. Group Size Ablation for INT4

Smaller group sizes mean more scale parameters (overhead) but better accuracy.
Let's find the sweet spot.

In [ ]:
group_sizes = [16, 32, 64, 128]
group_results = []

for gs in group_sizes:
    m = quantize_model_4bit(fp32_model, group_size=gs)
    m.eval()
    size = measure_model_size(m, f"INT4-g{gs}")
    ppl = compute_perplexity(m, eval_inputs, eval_targets, device="cpu")
    group_results.append({"group_size": gs, "size_mb": size, "perplexity": ppl})
    print(f"  Group size {gs}: PPL={ppl:.2f}, Size={size:.2f} MB")
    del m

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gs_vals = [r["group_size"] for r in group_results]
ppl_vals = [r["perplexity"] for r in group_results]
size_vals = [r["size_mb"] for r in group_results]

axes[0].plot(gs_vals, ppl_vals, 'bo-', markersize=8)
axes[0].axhline(y=fp32_ppl, color='red', linestyle='--', label=f'FP32 ({fp32_ppl:.2f})')
axes[0].set_xlabel('Group Size')
axes[0].set_ylabel('Perplexity')
axes[0].set_title('INT4 Perplexity vs Group Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(gs_vals, size_vals, 'go-', markersize=8)
axes[1].axhline(y=fp32_size, color='red', linestyle='--', label=f'FP32 ({fp32_size:.2f} MB)')
axes[1].set_xlabel('Group Size')
axes[1].set_ylabel('Model Size (MB)')
axes[1].set_title('INT4 Model Size vs Group Size')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('INT4 Group Size Ablation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Quantization Error Visualization by Layer

Different layers have different sensitivity to quantization. Let's visualize
which layers lose the most quality.

In [ ]:
# Compute per-layer quantization error
layer_errors_int8 = []
layer_errors_int4 = []
layer_names = []

for name, param in fp32_model.named_parameters():
    if param.dim() < 2:  # Skip 1D params (norms)
        continue

    w = param.data

    # INT8 error
    scale_8 = w.abs().amax(dim=1, keepdim=True) / 127
    scale_8 = scale_8.clamp(min=1e-8)
    w_q8 = (w / scale_8).round().clamp(-127, 127)
    w_deq8 = w_q8 * scale_8
    err8 = (w - w_deq8).pow(2).mean().sqrt().item()

    # INT4 error
    scale_4 = w.abs().amax(dim=1, keepdim=True) / 7
    scale_4 = scale_4.clamp(min=1e-8)
    w_q4 = (w / scale_4).round().clamp(-8, 7)
    w_deq4 = w_q4 * scale_4
    err4 = (w - w_deq4).pow(2).mean().sqrt().item()

    layer_errors_int8.append(err8)
    layer_errors_int4.append(err4)
    layer_names.append(name.replace("layers.", "L").replace(".weight", ""))

fig, ax = plt.subplots(figsize=(16, 6))
x = range(len(layer_names))
width = 0.35
ax.bar([i - width/2 for i in x], layer_errors_int8, width, label='INT8 RMSE', color='steelblue')
ax.bar([i + width/2 for i in x], layer_errors_int4, width, label='INT4 RMSE', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(layer_names, rotation=90, fontsize=7)
ax.set_ylabel('RMSE')
ax.set_title('Quantization Error by Layer')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 12. Generation Quality Comparison

In [ ]:
INSTRUCTION_PREFIX = "<|instruction|> "
RESPONSE_PREFIX = " <|response|> "

def generate_response(model, tokenizer, instruction, max_tokens=80):
    prompt = INSTRUCTION_PREFIX + instruction + RESPONSE_PREFIX
    input_ids = torch.tensor([tokenizer.encode(prompt)])
    output_ids = model.generate(input_ids, max_new_tokens=max_tokens, temperature=0.7, top_k=40)
    text = tokenizer.decode(output_ids[0].tolist())
    if "<|response|>" in text:
        resp = text.split("<|response|>")[-1]
        if "<|end|>" in resp:
            resp = resp.split("<|end|>")[0]
        return resp.strip()
    return text[len(prompt):].strip()


test_prompts = [
    "Write a short story about a rabbit.",
    "Tell a story about the ocean.",
]

models_to_test = [
    ("FP32", fp32_model),
    ("INT8-Dynamic", int8_dynamic_model),
    ("INT4", int4_model),
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("=" * 70)
    for name, model in models_to_test:
        resp = generate_response(model, tokenizer, prompt)
        print(f"  [{name}] {resp[:150]}")
    print()

## 13. Comprehensive Comparison Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = list(results.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

# Model size
sizes = [results[m]["size_mb"] for m in methods]
bars = axes[0].bar(methods, sizes, color=colors[:len(methods)])
axes[0].set_ylabel('Size (MB)')
axes[0].set_title('Model Size')
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Perplexity
ppls = [results[m]["perplexity"] for m in methods]
bars = axes[1].bar(methods, ppls, color=colors[:len(methods)])
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Quality (Lower is Better)')
for bar, val in zip(bars, ppls):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Speed
speeds = [results[m]["tokens_per_sec"] for m in methods]
bars = axes[2].bar(methods, speeds, color=colors[:len(methods)])
axes[2].set_ylabel('Tokens/Second')
axes[2].set_title('Inference Speed')
for bar, val in zip(bars, speeds):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.0f}', ha='center', va='bottom', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Quantization Comparison Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\nSummary Table:")
print(f"{'Method':<18} {'Size (MB)':>10} {'Perplexity':>12} {'Tok/s':>10} {'Size Ratio':>12}")
print("-" * 65)
for m in methods:
    r = results[m]
    ratio = fp32_size / r['size_mb']
    print(f"{m:<18} {r['size_mb']:>10.2f} {r['perplexity']:>12.2f} {r['tokens_per_sec']:>10.1f} {ratio:>11.2f}x")

## 14. Save Quantized Models

In [ ]:
# Save the INT8 dynamic model state
torch.save({
    "model_state_dict": int8_dynamic_model.state_dict(),
    "config": config,
    "quantization": "dynamic_int8",
    "perplexity": int8_dyn_ppl,
}, save_dir / "quantized_int8.pt")

# Save the INT4 model state
torch.save({
    "model_state_dict": int4_model.state_dict(),
    "config": config,
    "quantization": "int4_group32",
    "perplexity": int4_ppl,
}, save_dir / "quantized_int4.pt")

# Also save the FP32 model for serving (next notebook)
torch.save({
    "model_state_dict": fp32_model.state_dict(),
    "config": config,
    "stage": "dpo",
}, save_dir / "model_for_serving.pt")

print("\nSaved models:")
for f in sorted(save_dir.glob("*.pt")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

## Summary

In this notebook we:

1. **Analyzed** weight distributions to understand quantizability
2. **Applied** dynamic INT8 quantization (PyTorch native)
3. **Implemented** manual INT8 per-channel quantization from scratch
4. **Implemented** INT4 per-group quantization (GPTQ-style)
5. **Benchmarked** size, speed, and quality for each method
6. **Visualized** per-layer quantization sensitivity

Key findings:
- INT8 achieves ~2-4x compression with minimal quality loss
- INT4 achieves ~4-8x compression with moderate quality loss
- Smaller group sizes improve INT4 accuracy at the cost of overhead
- First/last layers tend to be more sensitive to quantization

**Checkpoint locations:** `checkpoints/quantized_int8.pt`, `checkpoints/quantized_int4.pt`